In [23]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

In [24]:
loader = PyPDFLoader('../data/data_science_syllabus.pdf')
docs = loader.load()
len(docs)

10

In [25]:
# Splitting the data in Docs
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_data = splitter.split_documents(docs)


In [26]:
# Usint the EMbeddings
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-2-preview')

# Creating the vector and storing it to vector DB
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

# Defining the llm
llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')


In [27]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"
    return {
        "context":context,
        "question":query
    }

In [28]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answers based on the context and the question provided to you.
    If you don't know the answer or didn't find anything in the context , then you can say 'I don't Know.'
    Context:[{context}]
    Question:[{question}]
""")

In [29]:
rag_chain = get_context | prompt | llm

res = rag_chain.invoke("Tell me about machine learning Course")


In [30]:
res.content

[{'type': 'text',
  'text': 'Based on the provided context, the machine learning part of the course is divided into two modules:\n\n**Module 6: Machine Learning – I (Supervised Learning)** (Duration: 4 weeks / Month 6)\n* **Topics:** \n  * ML pipeline\n  * Regression: Linear, Logistic\n  * Decision Tree, Random Forest, KNN\n  * Train-test split, model evaluation\n* **Tools:** Scikit-learn, Google Colab, ChatGPT, PyCaret (optional)\n* **Mini Projects:** Loan Approval or House Price Prediction, Predict Diabetes from health dataset\n* **Mock Interview:** Supervised Learning Models & Metrics\n\n**Module 7: Machine Learning – II (Unsupervised & Feature Engineering)** (Duration: 3 weeks / Month 7)\n* **Topics:** \n  * KMeans Clustering\n  * Dimensionality Reduction: PCA\n  * Feature selection, encoding, scaling\n  * Model tuning (GridSearchCV)\n* **Tools:** Scikit-learn, Seaborn, Colab',
  'extras': {'signature': 'EmAKXgFpFH0TUoaCACu90rTvS9/gqaLg/fmztI4i5ozDVpT3VQC2mSQp1GeUgJK8rHAPgsPwP7CnJ3